In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────
!pip install faiss-cpu sentence-transformers numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.9 MB/s eta 0:00:00


In [ ]:
# ── Cell 2: Imports & sanity check ──────────────────────────────
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from pathlib import Path

print("✓ faiss version   :", faiss.__version__)
print("✓ numpy version   :", np.__version__)

# CPU is fine — no GPU needed for Steps 1 & 2
import torch
print("✓ CUDA available  :", torch.cuda.is_available(), "(not required here)")

✓ faiss version   : 1.13.2
✓ numpy version   : 2.0.2
✓ CUDA available  : False (not required here)


- Cell 3 — Dummy knowledge base (Step 2 starts here)

In [ ]:
# ── Cell 3: Dummy knowledge base ────────────────────────────────
# These 15 entries are realistic but hand-crafted.
# When Ram shares the dataset (Day 3), you will update case_id,
# sequence_id, and descriptions to match real URFD sequences.
# When Ram shares pose JSONs (Day 5), update pose_summary values.

dummy_cases = [
    # ── FALL cases (8) ──────────────────────────────────────────
    {
        "case_id": "fall_001",
        "sequence_id": "fall-01-cam0",          # placeholder — update with real URFD id
        "label": "FALL",
        "description": "Person walking forward, sudden loss of balance, rapid descent to floor face-down.",
        "pose_summary": "torso_angle: 82°, hip_height: 0.15, velocity: high",
        "key_features": ["horizontal_body", "rapid_descent", "arms_extended"],
        "outcome": "FALL",
        "notes": "Classic forward fall, hands did not brace in time."
    },
    {
        "case_id": "fall_002",
        "sequence_id": "fall-02-cam0",
        "label": "FALL",
        "description": "Elderly person slips on flat surface, lateral fall to the right side.",
        "pose_summary": "torso_angle: 75°, hip_height: 0.18, velocity: medium",
        "key_features": ["lateral_collapse", "hip_impact", "arm_bracing"],
        "outcome": "FALL",
        "notes": "Side fall, hip contact with floor."
    },
    {
        "case_id": "fall_003",
        "sequence_id": "fall-03-cam0",
        "label": "FALL",
        "description": "Person stumbles at edge of mat, trips and falls forward onto knees then floor.",
        "pose_summary": "torso_angle: 70°, hip_height: 0.20, velocity: high",
        "key_features": ["trip", "forward_fall", "knee_impact"],
        "outcome": "FALL",
        "notes": "Tripping fall with sequential knee then torso contact."
    },
    {
        "case_id": "fall_004",
        "sequence_id": "fall-04-cam0",
        "label": "FALL",
        "description": "Person loses balance while standing still, slow backward fall.",
        "pose_summary": "torso_angle: 78°, hip_height: 0.16, velocity: low",
        "key_features": ["backward_fall", "slow_descent", "no_bracing"],
        "outcome": "FALL",
        "notes": "Slow-motion backward fall, often missed by detectors."
    },
    {
        "case_id": "fall_005",
        "sequence_id": "fall-05-cam0",
        "label": "FALL",
        "description": "Person collapses suddenly from upright position, syncope-like event.",
        "pose_summary": "torso_angle: 88°, hip_height: 0.10, velocity: very_high",
        "key_features": ["sudden_collapse", "no_warning", "full_floor_contact"],
        "outcome": "FALL",
        "notes": "Syncope-type fall, very abrupt."
    },
    {
        "case_id": "fall_006",
        "sequence_id": "fall-06-cam0",
        "label": "FALL",
        "description": "Person reaches for object on shelf, loses balance and falls sideways.",
        "pose_summary": "torso_angle: 65°, hip_height: 0.22, velocity: medium",
        "key_features": ["reaching_motion", "lateral_fall", "off_balance"],
        "outcome": "FALL",
        "notes": "Fall triggered by reaching motion."
    },
    {
        "case_id": "fall_007",
        "sequence_id": "fall-07-cam0",
        "label": "FALL",
        "description": "Person walking, right knee buckles, spiraling fall to the ground.",
        "pose_summary": "torso_angle: 60°, hip_height: 0.17, velocity: medium",
        "key_features": ["knee_buckle", "spiral_fall", "rotation"],
        "outcome": "FALL",
        "notes": "Rotational component makes detection harder."
    },
    {
        "case_id": "fall_008",
        "sequence_id": "fall-08-cam0",
        "label": "FALL",
        "description": "Person attempts to sit on chair, misses seat, falls backward.",
        "pose_summary": "torso_angle: 72°, hip_height: 0.19, velocity: medium",
        "key_features": ["missed_seat", "backward_fall", "chair_present"],
        "outcome": "FALL",
        "notes": "Common in elderly — mistimed sit."
    },

    # ── ADL (Activities of Daily Living) cases (7) ──────────────
    {
        "case_id": "adl_001",
        "sequence_id": "adl-01-cam0",
        "label": "NO_FALL",
        "description": "Person bends down to pick up object from floor, returns upright.",
        "pose_summary": "torso_angle: 55°, hip_height: 0.35, velocity: low",
        "key_features": ["controlled_bend", "upright_recovery", "slow_motion"],
        "outcome": "NO_FALL",
        "notes": "Common false positive trigger — torso tilts but controlled."
    },
    {
        "case_id": "adl_002",
        "sequence_id": "adl-02-cam0",
        "label": "NO_FALL",
        "description": "Person sits down slowly on chair in a controlled manner.",
        "pose_summary": "torso_angle: 40°, hip_height: 0.45, velocity: very_low",
        "key_features": ["controlled_descent", "seated_posture", "no_impact"],
        "outcome": "NO_FALL",
        "notes": "Descent is intentional, hip height remains moderate."
    },
    {
        "case_id": "adl_003",
        "sequence_id": "adl-03-cam0",
        "label": "NO_FALL",
        "description": "Person lies down on mat deliberately, slowly transitioning to floor.",
        "pose_summary": "torso_angle: 80°, hip_height: 0.12, velocity: very_low",
        "key_features": ["intentional_floor_contact", "slow_velocity", "controlled"],
        "outcome": "NO_FALL",
        "notes": "Horizontal body but velocity is very low — no fall."
    },
    {
        "case_id": "adl_004",
        "sequence_id": "adl-04-cam0",
        "label": "NO_FALL",
        "description": "Person walking normally across room, no instability observed.",
        "pose_summary": "torso_angle: 10°, hip_height: 0.65, velocity: medium",
        "key_features": ["upright_posture", "stable_gait", "normal_hip_height"],
        "outcome": "NO_FALL",
        "notes": "Normal walking baseline."
    },
    {
        "case_id": "adl_005",
        "sequence_id": "adl-05-cam0",
        "label": "NO_FALL",
        "description": "Person crouches to tie shoelace, stays low briefly, then stands.",
        "pose_summary": "torso_angle: 50°, hip_height: 0.30, velocity: low",
        "key_features": ["crouch", "low_hip", "controlled_return"],
        "outcome": "NO_FALL",
        "notes": "Low hip height but intentional and controlled."
    },
    {
        "case_id": "adl_006",
        "sequence_id": "adl-06-cam0",
        "label": "NO_FALL",
        "description": "Person waves arms while talking, slight body sway but stays upright.",
        "pose_summary": "torso_angle: 15°, hip_height: 0.62, velocity: low",
        "key_features": ["arm_motion", "slight_sway", "upright"],
        "outcome": "NO_FALL",
        "notes": "Arm motion can confuse pose estimation."
    },
    {
        "case_id": "adl_007",
        "sequence_id": "adl-07-cam0",
        "label": "NO_FALL",
        "description": "Person exercises — performs a squat, controlled descent and ascent.",
        "pose_summary": "torso_angle: 35°, hip_height: 0.38, velocity: medium",
        "key_features": ["squat", "hip_descent", "controlled_recovery"],
        "outcome": "NO_FALL",
        "notes": "Exercise motion — hip drops but velocity and recovery are controlled."
    },
]

print(f"✓ Knowledge base created: {len(dummy_cases)} cases")
print(f"  FALL    : {sum(1 for c in dummy_cases if c['label'] == 'FALL')}")
print(f"  NO_FALL : {sum(1 for c in dummy_cases if c['label'] == 'NO_FALL')}")

✓ Knowledge base created: 15 cases
  FALL    : 8
  NO_FALL : 7


- Cell 4 — Build FAISS index (Step 1 core)

In [ ]:
# ── Cell 4: Build FAISS index ────────────────────────────────────
# Load the embedding model (downloads ~90MB on first run)
print("Loading SentenceTransformer model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✓ Model loaded")

def build_faiss_index(cases):
    """
    Encodes each case as a text embedding and builds a FAISS L2 index.
    Returns: (index, embeddings, texts)
    """
    # Combine description + pose_summary as the searchable text per case
    texts = [
        f"{c['description']} {c['pose_summary']} {' '.join(c['key_features'])}"
        for c in cases
    ]

    print(f"Encoding {len(texts)} cases...")
    embeddings = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    print(f"✓ Embeddings shape: {embeddings.shape}")  # (N, 384) for MiniLM

    # Build flat L2 index (exact search — fine for <1000 cases)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype(np.float32))
    print(f"✓ FAISS index built: {index.ntotal} vectors indexed")

    return index, embeddings, texts

index, embeddings, texts = build_faiss_index(dummy_cases)

Loading SentenceTransformer model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Model loaded
Encoding 15 cases...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Embeddings shape: (15, 384)
✓ FAISS index built: 15 vectors indexed


- Cell 5 — Retrieval function + 3 test queries

In [ ]:
# ── Cell 5: Query function + demo ────────────────────────────────
def query_index(query_text, index, cases, k=3):
    """
    Retrieves top-k most similar cases for a given query.
    Returns list of (case, distance) tuples.
    """
    query_vec = embed_model.encode([query_text], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(query_vec, k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append((cases[idx], float(dist)))
    return results

def print_retrieval_results(query, results):
    print(f"\n{'='*60}")
    print(f"QUERY : {query}")
    print(f"{'='*60}")
    for rank, (case, dist) in enumerate(results, 1):
        print(f"\n  Rank {rank} | {case['label']} | distance: {dist:.4f}")
        print(f"  Case ID    : {case['case_id']}")
        print(f"  Description: {case['description']}")
        print(f"  Pose       : {case['pose_summary']}")
        print(f"  Notes      : {case['notes']}")

# ── 3 demo queries for your Slide 8 ─────────────────────────────
test_queries = [
    "Person suddenly collapses to the floor, horizontal body, rapid motion",
    "Person bends down slowly and picks something up from the ground",
    "Person misses chair and falls backward unexpectedly"
]

for q in test_queries:
    results = query_index(q, index, dummy_cases, k=3)
    print_retrieval_results(q, results)


QUERY : Person suddenly collapses to the floor, horizontal body, rapid motion

  Rank 1 | FALL | distance: 0.3558
  Case ID    : fall_005
  Description: Person collapses suddenly from upright position, syncope-like event.
  Pose       : torso_angle: 88°, hip_height: 0.10, velocity: very_high
  Notes      : Syncope-type fall, very abrupt.

  Rank 2 | FALL | distance: 0.7562
  Case ID    : fall_001
  Description: Person walking forward, sudden loss of balance, rapid descent to floor face-down.
  Pose       : torso_angle: 82°, hip_height: 0.15, velocity: high
  Notes      : Classic forward fall, hands did not brace in time.

  Rank 3 | NO_FALL | distance: 0.8047
  Case ID    : adl_003
  Description: Person lies down on mat deliberately, slowly transitioning to floor.
  Pose       : torso_angle: 80°, hip_height: 0.12, velocity: very_low
  Notes      : Horizontal body but velocity is very low — no fall.

QUERY : Person bends down slowly and picks something up from the ground

  Rank 1 | NO